# Step 1 — Select the Sentinel-2 scene

Searches a STAC API for Sentinel-2 L2A scenes intersecting the study area, acquired
during the last `days_back` days with less than `cloud_cover_max` % cloud cover, and keeps
the one with the lowest cloud cover.

| | |
|---|---|
| Six-phase position | Selection / filtering |
| W1 Algae Bloom counterpart | `select-products-sentinel2` (`catalog: earth-search`) |
| Output | `stac_item`: the selected STAC Item (JSON file) |

Unlike `select-products-sentinel2`, which returns a list of Item URLs, this step returns the
Item itself: later steps read band hrefs and scene properties from it without a new query.

In [ ]:
import json
from datetime import datetime, timedelta

from pystac_client import Client

In [ ]:
# CWL type annotations (removed by ipython2cwl in the generated tool)
from typing import List, Optional

from ipython2cwl.iotypes import (
    CWLDirectoryPathOutput,
    CWLFilePathInput,
    CWLFilePathOutput,
    CWLFloatInput,
    CWLIntInput,
    CWLMetadata,
    CWLNamespaces,
    CWLRequirement,
    CWLStringInput,
)

In [ ]:
cwl_requirements: CWLRequirement = {
    "NetworkAccess": {"networkAccess": True},
    "ResourceRequirement": {"coresMin": 1, "ramMin": 512},
}

In [ ]:
cwl_metadata: CWLMetadata = {
    "s:softwareVersion": "0.1.0",
    "s:keywords": ["ospd", "mangrove", "sentinel-2", "stac"],
    "s:author": [{"class": "s:Person", "s:name": "Cameron Sajedi"}],
    "s:contributor": [
        {"class": "s:Person", "s:name": "Gérald Fenoy", "s:affiliation": "GeoLabs"}
    ],
    "s:codeRepository": "https://github.com/starling-foundries/KindGrove",
    "s:license": "https://spdx.org/licenses/CC-BY-NC-SA-4.0",
    "s:description": "Select the least cloudy Sentinel-2 L2A scene over a bounding box",
}

In [ ]:
cwl_namespaces: CWLNamespaces = {
    "s": "https://schema.org/",
}

## Inputs

In [ ]:
west: CWLFloatInput = 95.15
south: CWLFloatInput = 15.9
east: CWLFloatInput = 95.35
north: CWLFloatInput = 16.1
cloud_cover_max: CWLFloatInput = 20.0
days_back: CWLIntInput = 90
stac_api: Optional[CWLStringInput] = "https://earth-search.aws.element84.com/v1"
collection: Optional[CWLStringInput] = "sentinel-2-l2a"

## Search

In [ ]:
bbox = [west, south, east, north]
end_date = datetime.now()
start_date = end_date - timedelta(days=days_back)
print(f"Searching {collection} on {stac_api}")
print(f"Bounds: ({west}, {south}) to ({east}, {north}), max cloud cover {cloud_cover_max}%")

search = Client.open(stac_api).search(
    collections=[collection],
    bbox=bbox,
    datetime=f"{start_date.isoformat()}/{end_date.isoformat()}",
    query={"eo:cloud_cover": {"lt": cloud_cover_max}},
)
items = list(search.items())
print(f"Found {len(items)} scenes")
if len(items) == 0:
    raise ValueError(f"No scenes found with <{cloud_cover_max}% cloud cover")

## Keep the least cloudy scene

In [ ]:
best_item = min(items, key=lambda x: x.properties.get("eo:cloud_cover", 100))
print(f"Selected scene: {best_item.id} ({best_item.datetime:%Y-%m-%d})")
print(f"Cloud cover: {best_item.properties.get('eo:cloud_cover', float('nan')):.1f}%")

stac_item: CWLFilePathOutput = "scene_item.json"
with open(stac_item, "w") as f:
    json.dump(best_item.to_dict(), f, indent=2)
print(f"Saved: {stac_item}")